# Module 04 — Notebook 2: pandas DataFrames

## Learning Objectives

By the end of this notebook you will be able to:
- Create DataFrames from dicts and CSV files
- Inspect a DataFrame: `.head()`, `.shape`, `.columns`, `.dtypes`, `.describe()`
- Select columns and rows by label and position
- Filter rows with boolean indexing: `df[df["col"] > value]`
- Handle missing values: `.isnull()`, `.fillna()`, `.dropna()`
- Sort rows with `.sort_values()`

**Time:** ~20 minutes

In [ ]:
import sys
sys.path.insert(0, "../../")
from src.checks import check_equal, check_type, check_approx, check_length, check_contains
import numpy as np
import pandas as pd
from pathlib import Path

DATA_PATH = Path("../../data/synthetic/evaluation_results.csv")
print("pandas version:", pd.__version__)
print("CSV exists:    ", DATA_PATH.exists())

## 1. What is a DataFrame?

Think of a DataFrame as a **typed spreadsheet in memory** — rows are observations, columns are attributes. It's the pandas equivalent of a JavaScript `Array<object>`.

| JS concept | pandas equivalent |
|------------|------------------|
| `Array<object>` | `pd.DataFrame` |
| `results[0]` | `df.iloc[0]` |
| `results.map(r => r.score)` | `df["score"]` |
| `results.map(r => ({ model: r.model, score: r.score }))` | `df[["model", "score"]]` |
| `results.filter(r => r.score > 0.9)` | `df[df["score"] > 0.9]` |
| `results.length` | `len(df)` or `df.shape[0]` |
| `Object.keys(results[0])` | `df.columns` |

Each column is a **Series** — a 1D labeled array. `df["score"]` returns a Series; `df[["model", "score"]]` returns a DataFrame.

In [ ]:
# Creating a DataFrame from a dict — keys become column names
data = {
    "model": ["model-a-v1", "model-a-v2", "model-b-v1"],
    "task":  ["factual_accuracy", "factual_accuracy", "factual_accuracy"],
    "score": [0.92, 0.95, 0.71],
}
df_small = pd.DataFrame(data)
print(df_small)
print("\nshape:  ", df_small.shape)        # (3, 3) — rows, columns
print("columns:", list(df_small.columns))
print("dtypes:\n", df_small.dtypes)

In [ ]:
# Loading from a CSV file — one line
df = pd.read_csv(DATA_PATH)
print("Shape:  ", df.shape)         # (20, 5)
print("Columns:", list(df.columns))
print()
df.head()   # renders as a nice table in Jupyter

## 2. Inspecting a DataFrame

Before analyzing any dataset, run these to orient yourself:

| Method | What it shows |
|--------|---------------|
| `df.head(n)` | first n rows (default 5) |
| `df.tail(n)` | last n rows |
| `df.shape` | (rows, columns) as a tuple |
| `df.dtypes` | column names and their types |
| `df.describe()` | summary stats for numeric columns |
| `df.isnull().sum()` | count of missing values per column |

In [ ]:
df = pd.read_csv(DATA_PATH)

rows, cols = df.shape
print(f"{rows} rows, {cols} columns")

print("\n--- dtypes ---")
print(df.dtypes)

print("\n--- describe() ---")
print(df.describe())

print("\n--- null counts ---")
print(df.isnull().sum())
# 'notes' has 9 nulls — many rows have no note

## 3. Selecting Columns and Rows

```javascript
// JS: get one property from every object
results.map(r => r.score)                          // → array of scores

// JS: get multiple properties
results.map(r => ({ model: r.model, score: r.score }))
```

```python
# pandas: select one column → Series (1D)
df["score"]

# pandas: select multiple columns → DataFrame
df[["model", "score"]]    # note: double brackets
```

For rows by position, use `.iloc[i]` (like `results[i]` in JS).

In [ ]:
df = pd.read_csv(DATA_PATH)

# Single column → Series
scores = df["score"]
print(type(scores))         # <class 'pandas.core.series.Series'>
print(scores.head())

# Multiple columns → DataFrame
slim = df[["model", "task", "score"]]
print("\nSlim view:")
print(slim.head())

# Row by position — like results[0]
print("\nFirst row (iloc[0]):")
print(df.iloc[0])

# Slice of rows — like results.slice(3, 6)
print("\nRows 3–5:")
print(df.iloc[3:6])

## 4. Filtering Rows

Same idea as NumPy boolean indexing, but on a DataFrame:

```javascript
results.filter(r => r.score > 0.9)
results.filter(r => r.model === "model-a-v2")
results.filter(r => r.score > 0.9 && r.model === "model-a-v2")
```

```python
df[df["score"] > 0.9]
df[df["model"] == "model-a-v2"]
df[(df["score"] > 0.9) & (df["model"] == "model-a-v2")]
```

> Two things to remember when combining conditions:
> 1. Use `&` (not `and`) — `and` works on scalars, not arrays
> 2. Wrap each condition in parentheses — Python's operator precedence requires it

In [ ]:
df = pd.read_csv(DATA_PATH)

# Filter: scores above 0.9
high = df[df["score"] > 0.9]
print(f"Rows with score > 0.9: {len(high)}")
print(high[["model", "task", "score"]])

# Filter: specific model
a2 = df[df["model"] == "model-a-v2"]
print(f"\nmodel-a-v2 rows: {len(a2)}")

# Combined filter — note & and parentheses
high_a2 = df[(df["score"] > 0.9) & (df["model"] == "model-a-v2")]
print(f"\nHigh-scoring model-a-v2 rows: {len(high_a2)}")

# Filter: danger zone — very low scores
danger = df[df["score"] < 0.7]
print(f"\nScore < 0.7 rows: {len(danger)}")
print(danger[["model", "task", "score"]])

## 5. Missing Values

pandas uses `NaN` (Not a Number) as its sentinel for missing data — like `null` or `undefined` in JS but with special behavior: `NaN != NaN` is `True`, so you can't check for it with `==`.

The standard workflow:
1. Discover: `df.isnull().sum()`
2. Decide: keep with `.fillna()`, drop with `.dropna()`, or filter with `.notnull()`

In [ ]:
df = pd.read_csv(DATA_PATH)

print("Null counts:")
print(df.isnull().sum())
# notes has 9 nulls

# Filter to rows that have a note
has_note = df[df["notes"].notnull()]
print(f"\nRows with notes: {len(has_note)}")

# Fill missing notes with a placeholder string
df_filled = df.copy()
df_filled["notes"] = df_filled["notes"].fillna("(no note)")
print("\nNull count after fillna:", df_filled["notes"].isnull().sum())

# Drop rows with any null — useful but lossy (removes 9 rows here)
df_clean = df.dropna()
print(f"Rows after dropna: {len(df_clean)}")

## 6. Sorting

`.sort_values(by, ascending=True/False)` — like JS `.sort()` but non-destructive (returns a new DataFrame).

In [ ]:
df = pd.read_csv(DATA_PATH)

# Sort by score descending
top = df.sort_values("score", ascending=False).head(5)
print("Top 5 scores:")
print(top[["model", "task", "score"]].to_string(index=False))

# Sort by multiple columns
sorted_df = df.sort_values(["model", "score"], ascending=[True, False])
print("\nSorted by model then score descending:")
print(sorted_df[["model", "task", "score"]].head(8).to_string(index=False))

---
## Your Turn — Exercise 1: Load and Inspect

Load `evaluation_results.csv` into `df`. Then:
1. Store the number of rows in `n_rows` and columns in `n_cols`.
2. Store the number of null values in the `"notes"` column in `null_notes`.
3. Store the mean score rounded to **4 decimal places** in `mean_score`.

In [ ]:
from pathlib import Path
DATA_PATH = Path("../../data/synthetic/evaluation_results.csv")

# YOUR CODE HERE
df         = None   # pd.read_csv(DATA_PATH)
n_rows     = None   # number of rows
n_cols     = None   # number of columns
null_notes = None   # null count in "notes" column
mean_score = None   # mean of "score" column, rounded to 4 decimal places

In [ ]:
check_type(df, pd.DataFrame, "df is a DataFrame")
check_equal(n_rows, 20, "n_rows is 20")
check_equal(n_cols, 5, "n_cols is 5")
check_equal(null_notes, 9, "9 null notes")
check_approx(mean_score, 0.8225, tolerance=1e-4, label="mean_score")

---
## Your Turn — Exercise 2: Filter Rows

Using the `df` you loaded above:
1. Create `low_scorers`: rows where score is **strictly below 0.7**.
2. Store the count in `n_low`.
3. Create `a_models`: rows where model starts with `"model-a"` (either version).
   > **Hint:** `df["model"].str.startswith("model-a")` returns a boolean Series.
4. Store the mean score for `a_models` in `a_mean`, rounded to **3 decimal places**.

In [ ]:
# YOUR CODE HERE  (df is still in scope from Exercise 1)
low_scorers = None   # rows where score < 0.7
n_low       = None   # count of low_scorers
a_models    = None   # rows where model starts with "model-a"
a_mean      = None   # mean score of a_models, rounded to 3 decimal places

In [ ]:
check_type(low_scorers, pd.DataFrame, "low_scorers is a DataFrame")
check_equal(n_low, 2, "2 scores below 0.7")
check_equal(len(a_models), 10, "10 model-a rows (5 per version)")
check_approx(a_mean, 0.911, tolerance=1e-3, label="a_mean")

---
## Your Turn — Exercise 3: Sort and Select

Using `df`:
1. Sort by score **descending** and store the top row's model name in `top_model`.
2. Store the top row's task name in `top_task`.

> **Hint:** `.sort_values("score", ascending=False).iloc[0]["model"]`

In [ ]:
# YOUR CODE HERE
top_model = None   # model name with the highest score
top_task  = None   # task name of the highest-scoring row

In [ ]:
check_equal(top_model, "model-a-v2", "top model is model-a-v2")
check_equal(top_task, "harmful_refusal", "top task is harmful_refusal")

---
## Why This Matters for AI Research Engineering

Every evaluation pipeline produces a table of results, and `pd.read_csv()` turns it into a DataFrame in one line.

From there:
- `df[df["score"] < 0.7]` immediately surfaces failing conditions — the first query in any safety audit
- `df[df["task"] == "harmful_refusal"]` isolates the safety-critical task you care about
- `df["notes"].fillna("")` prevents crashes when processing optional annotation fields
- `df.sort_values("score").head(10)` gives you the worst-performing cases to review first

> **Rule of thumb:** run `df.isnull().sum()` on every new dataset before analyzing it. Missing data silently distorts statistics if you don't handle it first.

## Summary

| What | Code |
|------|------|
| Create from dict | `pd.DataFrame({"col": [...]})` |
| Load CSV | `pd.read_csv(path)` |
| First/last rows | `df.head(n)`, `df.tail(n)` |
| Dimensions | `df.shape` → `(rows, cols)` |
| Column types | `df.dtypes` |
| Summary stats | `df.describe()` |
| Missing values | `df.isnull().sum()` |
| Select column | `df["col"]` → Series |
| Select columns | `df[["a", "b"]]` → DataFrame |
| Row by position | `df.iloc[i]` |
| Filter rows | `df[df["col"] > x]` |
| Multi-condition | `df[(cond1) & (cond2)]` |
| Sort | `df.sort_values("col", ascending=False)` |
| Fill missing | `df["col"].fillna(val)` |
| Drop nulls | `df.dropna()` |

**Next:** Notebook 3 — `groupby` and aggregation, where you'll compute per-model and per-task statistics.